# 01 — The Big Picture: Plain-Python RAG Mapped to LangChain

**By the end of this notebook, you can build a complete RAG loop in plain Python (no frameworks) and map each step to its LangChain module.**

### Learning Objectives
- Draw the two-phase RAG architecture (Index Time vs Query Time).
- Implement sentence splitting, word-overlap retrieval, prompt assembly, and generation from scratch in pure Python.
- Map every hand-written function to its modern LangChain replacement.

### Prerequisites
- Notebook 00 (`00_why_rag.ipynb`)

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 01:26:29.307000 10640 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: Index Time vs. Query Time (No Magic Boxes!)

Before using any framework, let's peel back every layer in **pure Python**. Every RAG system splits work cleanly into two timelines:
1. **Index Time (Prepare once)**: Read files -> slice into sentences/chunks -> save searchable entries.
2. **Query Time (Run per user question)**: Score entries against question -> pick Top-2 -> inject into prompt -> generate cited reply!

| Pure Python Step We Write Today | LangChain Replacement (Notebooks 02–06) |
|---|---|
| `Path.read_text()` | Native Loaders / `Document(page_content, metadata)` |
| `split_into_sentences()` | `RecursiveCharacterTextSplitter` (`langchain_text_splitters`) |
| `word_overlap_score()` | `Embeddings` + `InMemoryVectorStore.as_retriever()` |
| `f"<context>..."` + `llm.invoke` | `ChatPromptTemplate` + LCEL / `LangGraph` |

```
INDEX TIME (Run Once) :  Handbook + FAQ ---> Split into Sentences ---> Stored Index List
                                                                              |
QUERY TIME (Per Ask)  :  User Question ---> Overlap Scorer ---> Top-2 Hits ---+--> Prompt --> LLM --> Answer!
```

## 3. Step-by-Step Build

### Step 1: Index Time — Read Corpus & Split Into Sentences (Plain Python)
Using pure Python only, let's read `company_handbook.txt` and `product_faq.md` into `(source, sentence)` tuples.

In [2]:
import re
raw_files = ["company_handbook.txt", "product_faq.md"]
index_entries = []
for fname in raw_files:
    text = (data_dir / fname).read_text(encoding="utf-8")
    for line in [l.strip() for l in text.splitlines() if len(l.strip()) > 20]:
        index_entries.append((fname, line))
print(f"Plain-Python index built cleanly: {len(index_entries)} sentence entries.")
        

Plain-Python index built cleanly: 49 sentence entries.


> **What just happened?**
> Even a single Python `list` of `(filename, line)` tuples is technically a tiny searchable index!

### Step 2: Write a Pure-Python Word-Overlap Retriever
Let's write `search_overlap(query, k=2)` that extracts words with regex, scores overlap, and returns top matching lines.

In [3]:
def search_overlap(query: str, k: int = 2):
    q_words = {w for w in re.findall(r"[a-z0-9_]+", query.lower()) if len(w) > 2}
    scored = []
    for src, txt in index_entries:
        t_words = set(re.findall(r"[a-z0-9_]+", txt.lower()))
        scored.append((len(q_words & t_words), src, txt))
    scored.sort(reverse=True)
    return scored[:k]

hits = search_overlap("Which ROS2 distributions are supported Humble Iron Jazzy?", k=4)
for sc, src, txt in hits: print(f"[score={sc}] ({src}) {txt}")
        

[score=5] (product_faq.md) ### Which ROS2 distributions are natively supported by Novasphere Edge Agent?
[score=3] (product_faq.md) Older ROS1 distributions (such as Noetic) require the legacy ROS-Bridge adapter container and are not natively supported.
[score=2] (product_faq.md) Novasphere Edge Agent v4+ natively supports three long-term/active ROS2 distributions on Ubuntu 22.04 and 24.04 ARM64/x86_64 targets:
[score=1] (product_faq.md) - **Humble Hawksbill**


> **What just happened?**
> Word-overlap found exact keywords quickly — though notice how naive overlap struggles on synonyms without embeddings!

### Step 3: Pass Top Retrieved Lines Into the Prompt & Call LLM
Now let's format the Top-4 lines with `[source]` labels inside `<context>` tags and generate an answer.

In [4]:
q = "How much does the Enterprise Plan cost per month?"
top_hits = search_overlap(q, k=3)
ctx = "\n".join(f"[{src}] {txt}" for _, src, txt in top_hits)
ans = llm.invoke(f"Answer ONLY from context.\n<context>\n{ctx}\n</context>\nQuestion: {q}").content
print("Retrieved Context:\n" + ctx)
print("\nGenerated Answer :", ans)
        

Retrieved Context:
[product_faq.md] ZeroGravity Automatic Failover is exclusively included in the **Enterprise Plan** ($4,200/month). If a primary edge gateway or cluster coordinator becomes unresponsive for more than **40 milliseconds**, ZeroGravity automatically transfers fleet state ownership to a standby peer node without dropping active safety heartbeats.
[company_handbook.txt] - Professional Plan: $1,850 per month. Supports up to 50 active nodes. Includes the HyperSync protocol, 90-day telemetry retention, and dedicated Slack Connect support.
[company_handbook.txt] - Enterprise Plan: $4,200 per month. Supports up to 200 active nodes. Includes ZeroGravity automatic failover, QuantumCache v4, 1-year telemetry retention, and 24/7/365 mission-critical support.

Generated Answer : Based on the provided context: [company_handbook.txt] - Enterprise Plan: $4,200 per month. Supports up to 200 active nodes. Includes ZeroGravity automatic failover, QuantumCache v4, 1-year telemetry retentio

> **What just happened?**
> Inspecting `Retrieved Context` BEFORE trusting `Generated Answer` is a golden debugging habit we will follow throughout the course.

### Step 4: See Where Pure Word-Overlap Fails (Motivation for LangChain Vectors)
What if a beginner asks `'How much is the monthly fee for the top tier?'` without saying `'Enterprise'` or `'Plan'`? Overlap misses synonyms!

In [5]:
synonym_q = "How much is the monthly fee for the largest commercial subscription?"
missed = search_overlap(synonym_q, k=2)
print("Word-overlap hits for synonym phrasing:")
for sc, src, txt in missed: print(f"score={sc} -> {txt[:80]}")
        

Word-overlap hits for synonym phrasing:
score=2 -> ZeroGravity Automatic Failover is exclusively included in the **Enterprise Plan*
score=2 -> ### What uptime SLA is guaranteed for the Enterprise Tier?


> **What just happened?**
> Because `'fee'` != `'cost'` and `'largest'` != `'Enterprise'` literally, naive overlap scores weakly. Neural embeddings (Notebook 04) solve this cleanly!

## 4. Try It Yourself
1. **Test travel meal allowance** — *Hint:* Call `search_overlap('daily meal reimbursement limit VP approval', k=2)` and generate answer.
2. **Map Python to LangChain** — *Hint:* Review the table in Section 2 and check which file teaches each component.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [6]:
h_meal = search_overlap("daily meal reimbursement VP approval", k=2)
ctx_m = "\n".join(f"[{s}] {t}" for _, s, t in h_meal)
print(llm.invoke(f"<context>\n{ctx_m}\n</context>\nQuestion: What is the daily meal limit without VP approval?").content)
        

Based on the provided context: [company_handbook.txt] - Travel Meal Reimbursement: Employees traveling on company business are reimbursed up to $150 per day for meals and non-alcoholic beverages without requiring prior VP approval. Meal expenses above $150 per day require written pre-approval from the VP of Finance.


## 5. Common Mistakes
- Jumping straight into heavy libraries before understanding what split, search, and prompt formatting actually do.
- Relying purely on exact keyword overlap when users rephrase concepts with synonyms.
- Returning LLM answers without printing retrieved hits to spot retrieval bugs.

## 6. Recap
- Every RAG app boils down to Index Time (prepare chunks) and Query Time (find top-k + ground prompt).
- A 10-line plain-Python loop proves there is zero mystery behind retrieval-augmented generation.
- LangChain replaces each brittle manual helper with composable `Document` loaders, splitters, vectors, and graphs.

## 7. Next Notebook
Next (`02_loading_documents.ipynb`): Replace `read_text()` with unified LangChain `Document` loaders for TXT, Markdown, PDFs, folders, and web pages!

## 8. Video Chapter Outline
1. **Index Time vs Query Time Diagram** — Visualizing the complete 2-phase workflow.
2. **Pure Python Indexer** — Reading files into plain sentence tuples.
3. **Naive Overlap Retriever** — Implementing top-k search in 7 lines.
4. **Grounded Generation + Inspecting Chunks** — Building good debugging habits early.
5. **When Naive Overlap Fails & LangChain Map** — Connecting plain code to Notebooks 02–06.